# EVE Ref market-history publication study

This notebook investigates how one market date's archive filled in over time:

1. Which regions appeared, and which continued receiving records?
2. When did The Forge stop receiving new records?
3. Which items and item-region pairs arrived late?
4. Did most of the economic value arrive early?

**Reading the results:** clock zero is the first valid saved archive observation. A record is an item-region-day key. Its first-observed time is an upper bound, not its exact source publication time. The latest snapshot is a comparison reference, not a completeness guarantee.

The supporting module handles file validation, snapshot comparisons, and coverage checks. The analytical Polars queries and Plotly charts remain here. Normal execution reads local evidence and cached references only.

### TLDR

1. What evidence do we have?
    - Observed the market date 2026-10-11
    - First snapshot on 2026-10-12 02:11 UTC (with only two records)
    - First substantial archive arriving at 2026-10-12 12:11 UTC (7081 records)
    - Last unchanged archive arrived 2026-10-15 07:11 UTC (55883 records)
    - Stopped observing 2026-10-15 08:00 UTC
2. Which regions appeared and how did they fill in?
    - The Forge was most traded region, accumulating the vast majority of the region-type pairs
    - Most of the traded items after the 24 hour arrive in all other regions
3. When did The Forge stop receiving records?
    - Vast majority of Forge items arrived before the 24 hour mark
    - Last set of items arrived at the 28th hour mark
4. What was trickling in after the cutoff?
    - Just a few expensive supercarrier ships and miscellaneous niche items
5. Did the economically important records arrive early?
    - Yes.
    - 99.91% of the Forge records arrived by the 24 hour mark, which accounts for 99.98% of the total ISK traded on this market date

Did we observe any revisions/deletions in the data? No. This essentially confirms any potential suspicions in the data, especially based on the Everef code.

## Setup

Adjust the evidence directory, late-arrival boundary, and display limits below. Plotly is explicitly inline so running the notebook does not open browser tabs.

In [1]:
import sys
from datetime import timedelta
from itertools import pairwise
from pathlib import Path
from textwrap import wrap

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import polars as pl
from IPython.display import display
from plotly.subplots import make_subplots

pio.renderers.default = "plotly_mimetype"

In [2]:
def repository_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "experiments/everef/market_history_revisions/watch.py").exists():
            return candidate
    raise RuntimeError("Run from the repository root or a directory under experiments/.")


REPOSITORY_ROOT = repository_root()
EVIDENCE_DIR = REPOSITORY_ROOT / "experiments/.local/everef-market-history-revisions"
FORGE_REGION_ID = 10000002
LATE_CUTOFF_HOURS = 24
DETAIL_LIMIT = 10
RANKING_LIMIT = 10
ACQUIRE_REFERENCE_NAMES = False

# Support imports also work when the kernel starts beneath experiments/.
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

In [3]:
from experiments.everef.market_history_revisions.evidence import (
    KEY_FIELDS,
    POLL_TOLERANCE,
    cutoff_coverage,
    format_utc,
    load_reference_data,
    load_revision_evidence,
    parse_utc,
    threshold_table,
)


def preview(frame: pl.DataFrame, title: str) -> None:
    print(f"{title}: {frame.height:,} rows")
    display(frame.head(DETAIL_LIMIT))
    if frame.height > DETAIL_LIMIT:
        print(f"{frame.height - DETAIL_LIMIT:,} rows omitted; the complete frame remains available.")


def format_chart(figure: go.Figure, *, width: int = 850, height: int = 420) -> None:
    figure.update_layout(
        width=width,
        height=height,
        autosize=False,
        margin={"l": 70, "r": 30, "t": 80, "b": 100},
        legend={"orientation": "h", "yanchor": "top", "y": -0.25, "x": 0},
    )
    figure.update_xaxes(automargin=True, nticks=8)
    figure.update_yaxes(automargin=True)

In [4]:
study = load_revision_evidence(EVIDENCE_DIR)
references = load_reference_data(EVIDENCE_DIR, acquire=ACQUIRE_REFERENCE_NAMES)

baseline_at = study.baseline_at
snapshot_summary = study.snapshot_summary
observed_effects = study.observed_effects

## 1. What evidence do we have?

The watcher state is read once to compare adjacent usable archives. An unchanged hourly poll still contributes observation coverage; a long interval between downloads alone is not a gap. Invalid archives break comparisons but retain the first-seen ledger.

In [5]:
print(f"Market date: {study.market_date} | Hourly observations: {study.timeline.height:,}")
if baseline_at is not None:
    print(f"Baseline: {format_utc(baseline_at)} | Latest comparable poll: {format_utc(study.latest_comparable_at)}")
    print(f"Latest snapshot records: {study.latest_snapshot.height:,}")
else:
    print("No usable baseline yet. Continue watch.py and rerun the notebook.")

Market date: 2026-10-11 | Hourly observations: 93
Baseline: 2026-10-12T02:11:03.360105Z | Latest comparable poll: 2026-10-15T08:11:03.380567Z
Latest snapshot records: 55,883


In [6]:
for title, diagnostics in (
    ("Evidence diagnostics", study.diagnostics),
    ("Reference diagnostics", references.diagnostics),
    ("Invalid archives", study.invalid_snapshots),
):
    if not diagnostics.is_empty():
        preview(diagnostics, title)

totals_discrepancies = study.totals_review.filter(pl.col("outcome") != "matched")
for outcome in totals_discrepancies.group_by("outcome").len().sort("outcome").iter_rows(named=True):
    print(f"Totals review: {outcome['outcome']} ({outcome['len']:,} observations)")

Totals review: superseded_uncertain (1 observations)


`superseded_uncertain` simply means that the downloaded archive at `2026-10-13T03:11:03` initially had a downloaded archive row count that was different from the reported `totals.json`. It's a minor discrepancy that won't affect any of this analysis in this notebook or this project as a whole.

### Prepare named records and arrival cohorts

`events` contains the observed history, including removed records. `cohorts` contains only records still present in the latest snapshot, joined to their first-observed times. Keeping those populations separate avoids confusing publication history with current economic weight. Unresolved reference names stay explicit.

In [7]:
resolved_names = [
    pl.coalesce([pl.col("type_name"), pl.format("Unknown type {}", "type_id")]).alias("type_name"),
    pl.coalesce([pl.col("region_name"), pl.format("Unknown region {}", "region_id")]).alias("region_name"),
    pl.coalesce([pl.col("group_name"), pl.lit("Unknown group/unresolved")]).alias("group_name"),
    pl.coalesce([pl.col("category_name"), pl.lit("Unknown category/unresolved")]).alias("category_name"),
]

events = (
    study.events.join(references.types, on="type_id", how="left")
    .join(references.regions, on="region_id", how="left")
    .with_columns(resolved_names)
)
event_counts = events.group_by("kind").len().sort("kind")
preview(event_counts, "Arrivals, removals, reappearances, and revisions")

Arrivals, removals, reappearances, and revisions: 3 rows


kind,len
str,u32
"""added""",55880
"""baseline""",2
"""first_appearance_after_invalid…",1


In [8]:
if study.latest_snapshot is None:
    latest_values = pl.DataFrame(
        schema={"date": pl.String, "region_id": pl.Int64, "type_id": pl.Int64, "latest_total_isk_traded": pl.Float64}
    )
else:
    latest_values = study.latest_snapshot.select(
        *KEY_FIELDS,
        (pl.col("average") * pl.col("volume")).alias("latest_total_isk_traded"),
    )

cohorts = (
    latest_values.join(study.first_seen, on=list(KEY_FIELDS), how="left")
    .join(references.types, on="type_id", how="left")
    .join(references.regions, on="region_id", how="left")
    .with_columns(
        *resolved_names,
        pl.col("first_appeared_valid_at").alias("observed_at"),
        pl.lit(True).alias("first_observed"),
    )
)
assert latest_values.join(study.first_seen.select(list(KEY_FIELDS)), on=list(KEY_FIELDS), how="anti").is_empty()
late_arrivals = cohorts.filter(pl.col("elapsed_hours") > LATE_CUTOFF_HOURS)
print(f"Latest-snapshot total ISK traded: {latest_values['latest_total_isk_traded'].sum():,.2f} ISK")

Latest-snapshot total ISK traded: 29,674,101,879,683.90 ISK


### The initial two-row archive

The first archive that appeared for the Oct 11 market date only had two records, while the next saved snapshot appeared 10 hours later, containing a batch of 7,081 records. This initial snapshot defined hour 0 and `baseline`.

In [9]:
baseline_records = (
    events.filter(pl.col("kind") == "baseline")
    .select(
        "observed_at",
        "date",
        "type_name",
        "type_id",
        "region_name",
        "region_id",
        "average",
        "highest",
        "lowest",
        "volume",
        "order_count",
        "http_last_modified",
    )
    .sort("region_id", "type_id")
)

print(f"Initial archive: {baseline_records.height:,} records")
if baseline_at is not None:
    print(f"Observed at: {format_utc(baseline_at)}")
print("These records define hour 0 and baseline-region membership.")

with pl.Config(tbl_cols=-1, tbl_rows=-1, fmt_str_lengths=60):
    display(baseline_records)

Initial archive: 2 records
Observed at: 2026-10-12T02:11:03.360105Z
These records define hour 0 and baseline-region membership.


observed_at,date,type_name,type_id,region_name,region_id,average,highest,lowest,volume,order_count,http_last_modified
str,str,str,i64,str,i64,f64,f64,f64,i64,i64,str
"""2026-10-12T02:11:03.360105Z""","""2026-10-11""","""Alumel-Wired Enduring Sensor Booster""",6159,"""Immensea""",10000025,490000.0,490000.0,490000.0,1,1,"""2026-10-11T11:03:15Z"""
"""2026-10-12T02:11:03.360105Z""","""2026-10-11""","""Zainou 'Deadeye' Rapid Launch RL-1001""",27096,"""Deklein""",10000035,999000.0,999000.0,999000.0,3,3,"""2026-10-11T11:03:15Z"""


My guess as to why these two items appeared so late is due to the `ExplorerRegionTypeSource` in the [Everef scraper](https://docs.everef.net/commands/scrape-market-history.html#sources). This scraping component seems to run regularly throughout the day to ensure full coverage of all types and region pairs, hence why these two entries have very low volume/orders in less popular regions.

In [10]:
hourly_events = (
    study.events.with_columns(pl.col("elapsed_hours").floor().cast(pl.Int64).alias("hour"))
    .group_by("hour", "kind")
    .len()
    .sort("hour")
)
if not hourly_events.is_empty():
    hourly_chart = go.Figure()
    for (kind,), counts in hourly_events.group_by("kind", maintain_order=True):
        hours = counts["hour"].to_list()
        intervals = [
            [
                format_utc(baseline_at + timedelta(hours=hour)),
                format_utc(baseline_at + timedelta(hours=hour + 1)),
                hour,
                hour + 1,
            ]
            for hour in hours
        ]
        hourly_chart.add_bar(
            name=kind,
            x=hours,
            y=counts["len"],
            customdata=intervals,
            hovertemplate=(
                "kind=%{fullData.name}<br>elapsed %{customdata[2]}-%{customdata[3]}h"
                "<br>UTC %{customdata[0]} to %{customdata[1]}<br>count=%{y}<extra></extra>"
            ),
        )
    format_chart(hourly_chart)
    hourly_chart.update_layout(title="Hourly arrivals and revisions", barmode="group")
    hourly_chart.update_xaxes(title_text="Hours since first archive observation")
    hourly_chart.update_yaxes(title_text="Records")
    hourly_chart.add_vline(x=LATE_CUTOFF_HOURS, line_dash="dash")
    hourly_chart.show()

The `first_appearance_after_invalid_gap` appears to be due to an HTTP error that the watcher encountered; not an issue.

## 2. Which regions appeared, and how did they fill in?

Use all historical first arrivals here, even if records were later removed. The heatmap shows each region's initial batch and continued fill at later polls; latest-snapshot economic weights are examined in section 5.

In [11]:
first_arrivals = events.filter(pl.col("first_observed"))

In [12]:
region_hour_counts = (
    first_arrivals.with_columns(pl.col("elapsed_hours").floor().cast(pl.Int64).alias("hour"))
    .group_by("region_name", "hour")
    .len()
)

if not region_hour_counts.is_empty():
    region_order = (
        first_arrivals.group_by("region_name")
        .agg(pl.col("elapsed_hours").min().alias("first_appearance_hours"))
        .sort("first_appearance_hours", "region_name")
        .get_column("region_name")
        .to_list()
    )
    hour_order = sorted(region_hour_counts["hour"].unique().to_list())

    heatmap_grid = (
        pl.DataFrame({"region_name": region_order})
        .with_row_index("region_order")
        .join(pl.DataFrame({"hour": hour_order}), how="cross")
        .join(region_hour_counts, on=["region_name", "hour"], how="left")
        .with_columns(pl.col("len").fill_null(0))
        .sort("region_order", "hour")
    )
    counts = heatmap_grid["len"].to_numpy().reshape(len(region_order), len(hour_order))

    hour_intervals = [
        [
            format_utc(baseline_at + timedelta(hours=hour)),
            format_utc(baseline_at + timedelta(hours=hour + 1)),
            hour,
            hour + 1,
        ]
        for hour in hour_order
    ]

    # Preset color-scale limits. Limits above the observed maximum are omitted.
    full_range_max = int(counts.max())
    arrival_limits = [
        limit
        for limit in [10, 25, 50, 100, 250, 500, 1000, 2500, 5000]
        if limit < full_range_max
    ]
    arrival_limits.append(full_range_max)

    slider_steps = [
        {
            "method": "restyle",
            "args": [
                {
                    "zmin": [0],
                    "zmax": [limit],
                    "zauto": [False],
                },
                [0],
            ],
            "label": str(limit) if limit < full_range_max else "Full range",
        }
        for limit in arrival_limits
    ]

    heatmap = go.Figure(
        go.Heatmap(
            x=hour_order,
            y=region_order,
            z=counts,
            zmin=0,
            zmax=full_range_max,
            zauto=False,
            colorbar={"title": "Arrivals", "thickness": 14},
            customdata=[hour_intervals for _ in region_order],
            hovertemplate=(
                "region=%{y}<br>elapsed %{customdata[2]}-%{customdata[3]}h"
                "<br>UTC %{customdata[0]} to %{customdata[1]}"
                "<br>arrivals=%{z}<extra></extra>"
            ),
        )
    )

    format_chart(
        heatmap,
        width=900,
        height=max(490, 24 * len(region_order) + 220),
    )
    heatmap.update_layout(
        title="First-observed arrivals by region and hour",
        margin={"l": 160, "r": 100, "t": 80, "b": 140},
        sliders=[
            {
                "active": len(slider_steps) - 1,
                "currentvalue": {"prefix": "Arrivals color-scale upper limit: "},
                "pad": {"t": 65},
                "steps": slider_steps,
            }
        ],
    )
    heatmap.update_xaxes(
        title_text="Hours since first archive observation",
        showgrid=False,
    )
    heatmap.update_yaxes(
        title_text=None,
        categoryorder="array",
        categoryarray=region_order,
        autorange="reversed",
        dtick=1,
        showgrid=False,
    )
    heatmap.add_vline(
        x=LATE_CUTOFF_HOURS,
        line_dash="dash",
        annotation_text=f"{LATE_CUTOFF_HOURS}h cutoff",
        annotation_position="top",
    )
    heatmap.update_layout(
        autosize=False,
        width=900,
        margin={
            "l": 160,
            "r": 100,
            "t": 80,
            "b": 140,
            "autoexpand": False,
        },
    )
    heatmap.update_xaxes(automargin=False)
    heatmap.update_yaxes(automargin=False)

    heatmap.show(config={"responsive": False})
else:
    print("No first-observed arrivals in the saved evidence.")

The vast majority of the traded items in EVE Online occur within The Forge region. Since the Everef scraper prioritizes the most traded region-item pairs, it's only natural that most of the items that arrive in the archive first is in The Forge.

More notably, with the dashed line denoting the 24 hour mark, it is clear that the vast majority of the items also obtained within the first 24 hours, even more so if the chart was shifted left by the aforementioned 10 hours.

## 3. When did The Forge stop receiving records?

Track first arrivals separately from revisions. A quiet period measures time after the last addition/reappearance or market-value revision through the latest comparable poll. Coverage gaps and metadata/removal events are reported separately; quietness does not prove that the archive is final.

In [13]:
forge_events = events.filter(pl.col("region_id") == FORGE_REGION_ID)
forge_cohorts = cohorts.filter(pl.col("region_id") == FORGE_REGION_ID)
forge_effects = observed_effects.filter(pl.col("region_id") == FORGE_REGION_ID)
forge_latest = latest_values.filter(pl.col("region_id") == FORGE_REGION_ID)

forge_arrivals = (
    forge_events.filter(pl.col("first_observed"))
    .group_by("observed_at", "elapsed_hours")
    .len()
    .sort("elapsed_hours")
    .with_columns(pl.col("len").cum_sum().alias("cumulative_first_arrivals"))
)
if not forge_arrivals.is_empty():
    forge_arrival_chart = px.line(
        forge_arrivals,
        x="elapsed_hours",
        y="cumulative_first_arrivals",
        hover_data=["observed_at"],
        line_shape="hv",
        title="The Forge: cumulative first arrivals",
    )
    format_chart(forge_arrival_chart)
    forge_arrival_chart.update_xaxes(title_text="Hours since first archive observation")
    forge_arrival_chart.update_yaxes(title_text="Cumulative first arrivals")
    forge_arrival_chart.add_vline(x=LATE_CUTOFF_HOURS, line_dash="dash")
    forge_arrival_chart.show()
else:
    print("No Forge arrivals in the saved evidence.")

In [14]:
forge_tail = forge_events.filter(pl.col("elapsed_hours") > LATE_CUTOFF_HOURS).join(
    cohorts.select(*KEY_FIELDS, "first_appeared_valid_at", "latest_total_isk_traded"),
    on=list(KEY_FIELDS),
    how="left",
)
preview(
    forge_tail.select(
        "observed_at",
        "elapsed_hours",
        "kind",
        "type_name",
        "latest_total_isk_traded",
    ),
    "Forge records observed after the cutoff",
)
forge_timeline = forge_tail.group_by("observed_at", "elapsed_hours", "kind").len().sort("elapsed_hours")
if not forge_timeline.is_empty():
    forge_tail_chart = px.scatter(
        forge_timeline,
        x="elapsed_hours",
        y="len",
        color="kind",
        hover_data=["observed_at"],
        title="The Forge: late-tail event timeline",
    )
    forge_tail_chart.update_traces(
        hovertemplate="kind=%{fullData.name}<br>elapsed=%{x}h<br>UTC=%{customdata[0]}<br>events=%{y}<extra></extra>"
    )
    format_chart(forge_tail_chart)
    forge_tail_chart.update_xaxes(title_text="Hours since first archive observation")
    forge_tail_chart.update_yaxes(title_text="Events")
    forge_tail_chart.add_vline(x=LATE_CUTOFF_HOURS, line_dash="dash")
    forge_tail_chart.show()

Forge records observed after the cutoff: 19 rows


observed_at,elapsed_hours,kind,type_name,latest_total_isk_traded
str,f64,str,str,f64
"""2026-10-13T02:11:03.362869Z""",24.000001,"""added""","""Ammatar Navy Sergeant Major In…",10668.0
"""2026-10-13T02:11:03.362869Z""",24.000001,"""added""","""Triglavian Trinary Datastream …",5.0
"""2026-10-13T02:11:03.362869Z""",24.000001,"""added""","""Triglavian Trinary Datastream …",6.36
"""2026-10-13T02:11:03.362869Z""",24.000001,"""added""","""Triglavian Trinary Datastream …",10.0
"""2026-10-13T02:11:03.362869Z""",24.000001,"""added""","""Triglavian Trinary Datastream …",318.12
"""2026-10-13T02:11:03.362869Z""",24.000001,"""added""","""Triglavian Trinary Datastream …",18.24
"""2026-10-13T02:11:03.362869Z""",24.000001,"""added""","""Triglavian Trinary Datastream …",56.0
"""2026-10-13T02:11:03.362869Z""",24.000001,"""added""","""Expired Enigmatic Warp Matrix …",0.02
"""2026-10-13T02:11:03.362869Z""",24.000001,"""added""","""F-24 Entangled GQD""",44.0


9 rows omitted; the complete frame remains available.


In [15]:
forge_additions = forge_events.filter(pl.col("kind").is_in(["added", "reappeared"]))
forge_revisions = forge_events.filter(pl.col("kind") == "market_values_modified")
forge_other_events = forge_events.filter(pl.col("kind").is_in(["removed", "metadata_only_modified"]))

if not forge_events.is_empty():
    last_addition = forge_additions["observed_at"].max()
    last_revision = forge_revisions["observed_at"].max()
    last_qualifying = max(
        value for value in (last_addition, last_revision, format_utc(baseline_at)) if value is not None
    )
    latest_at = format_utc(study.latest_comparable_at)
    quiet_polls = [
        parse_utc(value)
        for value in snapshot_summary.filter(pl.col("comparable"))["observed_at"]
        if parse_utc(value) >= parse_utc(last_qualifying)
    ]
    quiet_gap = any(end - start > POLL_TOLERANCE for start, end in pairwise(quiet_polls))
    quiet_hours = (parse_utc(latest_at) - parse_utc(last_qualifying)).total_seconds() / 3600
    print(f"Last Forge addition/reappearance: {last_addition or 'none'} | Market-value revision: {last_revision or 'none'}")
    print(f"Latest comparable poll: {latest_at} | {quiet_hours:.1f}h since last addition/revision")
    print(f"Quiet-period comparable polls: {len(quiet_polls):,} | Coverage gap: {quiet_gap}")
    if not forge_other_events.is_empty():
        print(f"Metadata/removal events across all Forge evidence: {forge_other_events.height:,}")
else:
    print("No Forge observations; no quietness claim.")

Last Forge addition/reappearance: 2026-10-13T06:11:03.363207Z | Market-value revision: none
Latest comparable poll: 2026-10-15T08:11:03.380567Z | 50.0h since last addition/revision
Quiet-period comparable polls: 37 | Coverage gap: True


As we can see, we actually only have a trickling trail of items coming in after the 24 hour mark, with the final few items arriving from The Forge region at the 28th hour.

## 4. What was trickling in after the cutoff?

These rankings cover **all regions**, using the latest snapshot's total ISK traded for currently present records first observed after the cutoff. Late describes when the record appeared, not when its trades happened. Stable IDs participate in grouping, while chart labels show readable names only. Complete rankings remain available; charts show only their leading entries.

In [16]:
def rank_late_arrivals(frame: pl.DataFrame, columns: list[str], label: str) -> pl.DataFrame:
    if frame.is_empty():
        return pl.DataFrame(schema={label: pl.String, "late_arrivals": pl.Int64, "latest_total_isk_traded": pl.Float64})
    name_columns = [column for column in columns if column.endswith("_name")]
    return (
        frame.group_by(columns, maintain_order=True)
        .agg(
            pl.len().alias("late_arrivals"),
            pl.col("latest_total_isk_traded").sum(),
        )
        .with_columns(
            pl.concat_str(
                [pl.col(column).cast(pl.String).fill_null("unresolved") for column in name_columns],
                separator=" | ",
            ).alias(label),
        )
        .sort("latest_total_isk_traded", descending=True)
    )


def show_late_ranking(ranking: pl.DataFrame, label: str, title: str) -> None:
    if not ranking.is_empty():
        # Separate plotting keys keep equal display names from overlapping.
        leaders = ranking.head(RANKING_LIMIT).with_row_index("rank").with_columns(pl.col("rank").cast(pl.String))
        ranking_chart = px.bar(
            leaders,
            x="latest_total_isk_traded",
            y="rank",
            orientation="h",
            title=title,
            labels={"latest_total_isk_traded": "Total ISK traded", "rank": ""},
            custom_data=[label, "late_arrivals"],
        )
        ranking_chart.update_traces(
            hovertemplate=(
                "%{customdata[0]}<br>Late arrivals=%{customdata[1]}"
                "<br>Total ISK traded=%{x:,.0f}<extra></extra>"
            )
        )
        format_chart(ranking_chart, height=max(420, 54 * leaders.height + 130))
        ranking_chart.update_layout(margin={"l": 300, "b": 70})
        ranking_chart.update_yaxes(
            categoryorder="array",
            categoryarray=leaders["rank"].to_list(),
            autorange="reversed",
            tickmode="array",
            tickvals=leaders["rank"].to_list(),
            ticktext=["<br>".join(wrap(value, width=42)) for value in leaders[label]],
        )
        ranking_chart.show()

### Individual item-region records

Start with the largest individual late arrivals. The same item arriving in two regions is two different records.

In [17]:
late_item_region_ranking = rank_late_arrivals(
    late_arrivals,
    ["type_id", "type_name", "region_id", "region_name"],
    "item_region",
)
show_late_ranking(late_item_region_ranking, "item_region", "Late-arriving item-region records: total ISK traded")

### Groups and categories

Reference-data groups and categories show which kinds of items account for the late economic weight. Unresolved classifications remain explicit.

In [18]:
late_group_ranking = rank_late_arrivals(late_arrivals, ["group_id", "group_name"], "group")
show_late_ranking(late_group_ranking, "group", "Late-arriving records by group: total ISK traded")

In [19]:
late_category_ranking = rank_late_arrivals(late_arrivals, ["category_id", "category_name"], "category")
show_late_ranking(late_category_ranking, "category", "Late-arriving records by category: total ISK traded")

There are only a small handful of item-region pairs that come in after 24 hours, with the vast majority of the total ISK traded ends up being rare, expensive Supercarrier ships (Aeon and Nyx). The other items are largely just some miscellaneous items, each with niche use cases.

## 5. Did economically important records arrive early?

**`total_isk_traded = average (VWAP) × volume`**. There are two complementary views:

- **Latest-current cohorts:** assign each currently present record's latest total ISK traded to its first-observed time. This measures the economic weight of early versus late arrivals.
- **As-observed signed effects:** accumulate baseline total ISK traded and the signed effects of additions, reappearances, removals, and revisions. This measures observed availability and can be nonmonotonic. Invalid comparison resets leave impacts unresolved. However, based on how Everef scrapes data, there are only ever additions to the data. This is here, just in case.

Both use the latest actual snapshot for their denominator. The cutoff uses observations at or before that boundary; hourly sampling cannot establish exact publication times between polls. A coverage gap crossing the boundary makes cutoff availability uncertain. The compact cutoff table reports latest-current cohort shares; the charts retain both views.

In [20]:
coverage = cutoff_coverage(snapshot_summary, baseline_at, LATE_CUTOFF_HOURS)
has_reset = snapshot_summary.filter(pl.col("note").str.contains("reset snapshot")).height > 0
impact_coverage = "unresolved_reset_impact" if has_reset else coverage

coverage_global = threshold_table(
    "Global",
    cohorts,
    observed_effects,
    latest_values,
    coverage,
    LATE_CUTOFF_HOURS,
    targets=(),
    impact_coverage=impact_coverage,
)
coverage_forge = threshold_table(
    "The Forge",
    forge_cohorts,
    forge_effects,
    forge_latest,
    coverage,
    LATE_CUTOFF_HOURS,
    targets=(),
    impact_coverage=impact_coverage,
)
if impact_coverage != coverage:
    print(f"Signed-impact coverage: {impact_coverage}")

Signed-impact coverage: unresolved_reset_impact


In [21]:
cutoff_summary = (
    pl.concat([coverage_global, coverage_forge])
    .filter(pl.col("view") == "latest_current_cohort")
    .group_by("scope", maintain_order=True)
    .agg(
        (pl.col("share").filter(pl.col("measure").str.starts_with("rows_share_at_")).first() * 100).alias(
            "record_share_pct"
        ),
        (pl.col("share").filter(pl.col("measure").str.starts_with("total_isk_traded_share_at_")).first() * 100).alias(
            "total_isk_traded_share_pct"
        ),
        pl.col("status").unique().sort().str.join("; ").alias("status"),
    )
)
with pl.Config(float_precision=4, tbl_cols=-1, fmt_str_lengths=60):
    preview(cutoff_summary, f"Latest-current coverage by {LATE_CUTOFF_HOURS}h (%)")

Latest-current coverage by 24h (%): 2 rows


scope,record_share_pct,total_isk_traded_share_pct,status
str,f64,f64,str
"""Global""",95.2597,99.5625,"""available_with_earlier_gap_caveat"""
"""The Forge""",99.8210,99.9789,"""available_with_earlier_gap_caveat"""


### Compare record coverage with economic coverage

Each scope has two panels: record share and total ISK traded share, with both valuation views overlaid. Steps represent observed batches rather than continuous publication. Tables and charts use the same numeric Float64 calculations.

In [22]:
def cumulative_share_curve(
    records: pl.DataFrame,
    latest_reference: pl.DataFrame,
    *,
    signed_effects: bool = False,
) -> pl.DataFrame:
    isk_column = "total_isk_traded_delta" if signed_effects else "latest_total_isk_traded"
    row_expression = pl.col("row_effect").sum() if signed_effects else pl.len()
    points = (
        records.group_by("observed_at", "elapsed_hours")
        .agg(
            row_expression.alias("batch_rows"),
            pl.col(isk_column).sum().alias("batch_total_isk_traded"),
        )
        .sort("elapsed_hours")
    )
    row_denominator = latest_reference.height
    isk_denominator = latest_reference["latest_total_isk_traded"].sum()
    return points.with_columns(
        (
            pl.col("batch_rows").cum_sum().truediv(row_denominator)
            if row_denominator
            else pl.lit(None, dtype=pl.Float64)
        ).alias("row_share"),
        (
            pl.col("batch_total_isk_traded").cum_sum() / isk_denominator
            if isk_denominator
            else pl.lit(None, dtype=pl.Float64)
        ).alias("total_isk_traded_share"),
    )


global_cohort_curve = cumulative_share_curve(cohorts, latest_values)
global_observed_curve = cumulative_share_curve(observed_effects, latest_values, signed_effects=True)
forge_cohort_curve = cumulative_share_curve(forge_cohorts, forge_latest)
forge_observed_curve = cumulative_share_curve(forge_effects, forge_latest, signed_effects=True)

In [23]:
for scope_name, cohort_curve, observed_curve in (
    ("Global", global_cohort_curve, global_observed_curve),
    ("The Forge", forge_cohort_curve, forge_observed_curve),
):
    if cohort_curve.is_empty() and observed_curve.is_empty():
        continue
    share_chart = make_subplots(rows=1, cols=2, subplot_titles=("Record share", "Total ISK traded share"))
    for view_name, curve, color in (
        ("Latest-current cohorts", cohort_curve, px.colors.qualitative.Plotly[0]),
        ("As-observed effects", observed_curve, px.colors.qualitative.Plotly[1]),
    ):
        for panel, measure in ((1, "row_share"), (2, "total_isk_traded_share")):
            share_chart.add_trace(
                go.Scatter(
                    x=curve["elapsed_hours"],
                    y=curve[measure],
                    name=view_name,
                    legendgroup=view_name,
                    showlegend=panel == 1,
                    line={"shape": "hv", "color": color},
                    customdata=curve["observed_at"],
                    hovertemplate="UTC %{customdata}<br>elapsed=%{x}h<br>share=%{y:.2%}<extra></extra>",
                ),
                row=1,
                col=panel,
            )
    format_chart(share_chart, width=1050, height=450)
    share_chart.add_vline(x=LATE_CUTOFF_HOURS, line_dash="dash")
    share_chart.update_xaxes(title_text="Hours since first archive observation")
    share_chart.update_yaxes(tickformat=".0%")
    share_chart.update_layout(title=f"{scope_name}: record and total ISK traded coverage")
    share_chart.show()

As we can see, for the Forge, more than 99.9% of the data appears within the first 24 hours, which accounts for 99.98% of the total ISK traded on this market date.

The other, less traded regions, generally account for the late accumulation.